#### **Pan-African Youth Employment Cockpit – Dataset Generation**

**Objective**: Generate realistic synthetic data that mirrors Mastercard Foundation's Young Africa Works programs across Kenya, Ethiopia, Nigeria, Rwanda, and Ghana (2022–2025).

**Key References**:
- Young Africa Works Strategy: Enable 30 million young Africans (70% young women) in dignified work by 2030.
- Focus sectors: Agribusiness, Digital Skills, Green Jobs, Hospitality.
- Strong emphasis on gender equity, skills-to-employment transition, and outcome tracking.

#### Imports & Setup

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

# Reproducibility
np.random.seed(42)
random.seed(42)

print("Environment ready")
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/reference', exist_ok=True)

Environment ready


#### Master Parameters

In [3]:
countries = ['Kenya', 'Ethiopia', 'Nigeria', 'Rwanda', 'Ghana']

sectors = ['Agribusiness', 'Digital Skills', 'Green Jobs', 'Retail & Hospitality']

program_dict = {
    'Kenya': ['Young Africa Works Kenya', 'Digital Jobs for Youth'],
    'Ethiopia': ['Young Africa Works Ethiopia', 'Skills for Employment and Productivity'],
    'Nigeria': ['Young Africa Works Nigeria', 'Agri-Youth Empowerment'],
    'Rwanda': ['Hanga Ahazaza (Create the Future)', 'Youth Skills for Tourism'],
    'Ghana': ['Young Africa Works Ghana', 'Youth Entrepreneurship Programme']
}

start_date = datetime(2022, 1, 1)
end_date = datetime(2025, 3, 31)

print(f"Generating data for {len(countries)} countries | Period: 2022 - Q1 2025")

Generating data for 5 countries | Period: 2022 - Q1 2025


#### Generate Participants Table

In [4]:
n_participants = 25_000 

data = {
    'participant_id': [f'YF_{str(i).zfill(8)}' for i in range(1, n_participants + 1)],
    'country': np.random.choice(countries, n_participants, p=[0.25, 0.22, 0.23, 0.15, 0.15]),
    'age': np.random.randint(18, 36, n_participants),
    'gender': np.random.choice(['Male', 'Female'], n_participants, p=[0.32, 0.68]),  # ~70% female target
    'education_level': np.random.choice(
        ['Secondary', 'TVET Certificate', 'Diploma', 'Bachelor'], 
        n_participants, p=[0.35, 0.40, 0.18, 0.07]
    ),
    'disability_status': np.random.choice(['No', 'Yes'], n_participants, p=[0.92, 0.08]),
    'registration_date': [
        start_date + timedelta(days=random.randint(0, (end_date - start_date).days)) 
        for _ in range(n_participants)
    ]
}

participants = pd.DataFrame(data)
participants['registration_date'] = pd.to_datetime(participants['registration_date'])
print(f"Participants table created: {len(participants):,} records")

Participants table created: 25,000 records


#### Generate Enrollments Table

In [5]:
enrollments = participants.copy()
enrollments = enrollments.rename(columns={'registration_date': 'enrollment_date'})

enrollments['sector'] = enrollments['country'].map(lambda c: random.choice(sectors))
enrollments['program_name'] = enrollments['country'].map(lambda c: random.choice(program_dict[c]))
enrollments['training_duration_days'] = np.random.choice([30, 60, 90, 120], len(enrollments), p=[0.2, 0.35, 0.3, 0.15])
enrollments['completion_status'] = np.random.choice(
    ['Completed', 'Dropped Out', 'In Progress'], 
    len(enrollments), 
    p=[0.78, 0.15, 0.07]
)

print(f"Enrollments table created: {len(enrollments):,} records")

Enrollments table created: 25,000 records


#### Generate Placements Table

In [7]:
placements = enrollments[enrollments['completion_status'] == 'Completed'].copy().reset_index(drop=True)

placements['placement_date'] = placements['enrollment_date'] + pd.to_timedelta(
    np.random.randint(15, 180, len(placements)), unit='D'
)

# Placement success
placements['placed'] = np.random.choice([0, 1], len(placements), p=[0.22, 0.78])

placed_mask = placements['placed'] == 1
n_placed = placed_mask.sum()

placements['employment_type'] = None
placements.loc[placed_mask, 'employment_type'] = np.random.choice(
    ['Formal Wage', 'Self-Employment', 'Informal'], 
    n_placed, 
    p=[0.45, 0.35, 0.20]
)

placements['monthly_income_usd'] = None
placements.loc[placed_mask, 'monthly_income_usd'] = np.random.normal(
    280, 120, n_placed
).clip(80, 850).astype(int)

print(f"Placements table created: {len(placements):,} records")
print(f"Successfully placed: {n_placed:,} participants")

Placements table created: 19,324 records
Successfully placed: 15,049 participants


#### Post-Placement Follow-up

In [8]:
follow_ups = placements[placements['placed'] == 1].copy().reset_index(drop=True)

follow_ups['followup_6m_date'] = follow_ups['placement_date'] + timedelta(days=180)
follow_ups['still_employed_6m'] = np.random.choice([0, 1], len(follow_ups), p=[0.28, 0.72])

np.random.seed(42)
issue_idx = follow_ups.sample(frac=0.03).index
follow_ups.loc[issue_idx, 'monthly_income_usd'] = np.nan

issue_idx2 = follow_ups.sample(frac=0.015).index
follow_ups.loc[issue_idx2, 'still_employed_6m'] = -1  

print(f"Post-placement follow-up table created: {len(follow_ups):,} records")
print("Intentional data quality issues injected for later governance phase")

Post-placement follow-up table created: 15,049 records
Intentional data quality issues injected for later governance phase


#### Save All Tables

In [9]:
participants.to_csv('../data/raw/participants.csv', index=False)
enrollments.to_csv('../data/raw/enrollments.csv', index=False)
placements.to_csv('../data/raw/placements.csv', index=False)
follow_ups.to_csv('../data/raw/post_placement_followups.csv', index=False)

print("All raw datasets saved successfully!")
print(f"Total participants: {len(participants):,}")

All raw datasets saved successfully!
Total participants: 25,000
